# 第十六课｜为什么搬数据比加法更难？

event machine 看起来主要在做加法与比较。但规模增大后，常见问题会变成：

> **数据还没送到计算单元，计算单元就在等。**

主要新概念：**内存层次与数据移动成本会限制吞吐。**

## 1. 概念账本

**已经知道：** neuron state、synapse records、host/PL、event-driven processing。

**今天学习：**
- **内存层次（memory hierarchy）**；
- **延迟（latency）**：一次操作从发起到结果可用要等多久；
- **吞吐（throughput）**：单位时间持续完成多少工作；
- **带宽（bandwidth）**：单位时间持续搬运多少数据。

**只预告：** DDR 与 AXI 分别在后两课学习。

## 2. 为什么这会出现在果蝇项目里？

source spike 到来后，需要读 source index 与 synapse records。加一个 weight 很便宜，但把大量 records 送到 engine 可能更慢。

```mermaid
flowchart LR
  MEM["synapse storage"] -->|bytes / time| ENG["synapse engine"]
  ENG --> ACC["target update"]
```

如果数据送不够快，再多算术资源也可能空闲。

## 3. 三个指标不要混为一谈

- latency：一次要等多久；
- throughput：持续运行时每单位时间完成多少件；
- bandwidth：持续运行时每单位时间搬多少 bytes。

低 latency 不自动等于高 bandwidth；高 bandwidth 也不保证单次 random access 很快。

## 4. 一个最小 cost model

先使用不重叠的简化模型：

- compute time = item 数 × 每 item 计算时间；
- transfer time = startup latency + 总 bytes ÷ bandwidth；
- 谁更大，就先把谁视为当前主要瓶颈。

真实硬件可以让计算与传输重叠，本课暂不引入 pipeline overlap。

## 5. Run：算术和搬运谁更慢？

先预测下面参数下哪一个时间更大。

In [ ]:
num_items = 1000
bytes_per_item = 16
compute_ns_per_item = 1.0
bandwidth_bytes_per_ns = 4.0
startup_latency_ns = 100.0

compute_ns = num_items * compute_ns_per_item
transfer_ns = startup_latency_ns + (num_items * bytes_per_item) / bandwidth_bytes_per_ns

print("compute time:", compute_ns, "ns")
print("data-movement time:", transfer_ns, "ns")
print("dominant cost:", "memory/data movement" if transfer_ns > compute_ns else "compute")


## 6. Observe

16,000 bytes 以 4 bytes/ns 搬运需要 4,000 ns，再加 100 ns startup；计算只需 1,000 ns。这个例子主要在等数据。

## 7. Try It

把 bandwidth 从 4.0 改成 32.0，预测 bottleneck 是否改变；再把每 item 计算时间调大，看何时转为 compute-bound。

## 8. 作业

[第 16 课作业：判断 compute 还是 data movement 限制系统](../../exercises/zh/16_data_movement_cost.ipynb)

## 9. AI Task

让 AI 根据事件数、每事件 bytes、bandwidth、compute time 解释瓶颈。你负责检查单位，并确认它没有把 latency 与 bandwidth 当成同一个量。

## 10. Human Check

解释 memory hierarchy 为什么会影响 event-driven SNN；latency、throughput、bandwidth 的区别；为什么“加法器很快”不能推出系统吞吐一定高。

## 11. Engineering Handoff

对应 `RMD-013A`：在接 DDR/AXI 之前建立 memory hierarchy 与 bandwidth 直觉，并使用可测量指标描述数据移动。

## 12. 项目追踪 Project Trace

- Lesson: `LSN-016`
- Mapping: `RMD-013A`
- Metrics: latency / throughput / bandwidth
- Boundary: no DDR protocol details yet

## 13. Exit Ticket

你能根据 compute 与 data-movement 参数判断当前更可能被哪一侧限制，并指出模型假设。